In [1]:
import os
import sys
sys.path.append(os.path.abspath("../../.."))

In [2]:
import pandas as pd
from classes.trading.actionPredictionTrading import ActionPredictionTrading

# Load the dataset
csv_path = "../../datasets/b3_dados/processed/acoes_concat.csv"
df = pd.read_csv(csv_path, parse_dates=['Date'])

test_start_date = '2019-01-01'

# Filtra o conjunto de teste
test_data = df[df['Date'] >= test_start_date]

In [3]:


# List of stocks and paths to models and scalers
stocks_models_scalers = {
    "ITUB4": {
        "model_path": "../../../saved_models/ITUB4_model_v1.1.pkl",
        "scaler_x_path": "../../../saved_models/ITUB4_scaler_X_v1.1.pkl"
    },
    "VALE3": {
        "model_path": "../../../saved_models/VALE3_model_v1.1.pkl",
        "scaler_x_path": "../../../saved_models/VALE3_scaler_X_v1.1.pkl"
    },
}


In [4]:
# Iterate over each stock
results = {}
for stock, paths in stocks_models_scalers.items():
    print(f"Analyzing stock: {stock}")
    analysis = ActionPredictionTrading(
        test_data, stock, model_path=paths["model_path"]
    )
    analysis.load_model()
    if "scaler_x_path" in paths:
        analysis.load_scaler(paths["scaler_x_path"])
    analysis.generate_predictions()

    # Executa as simulações
    result_no_stop = analysis.simulate_trading(stop_loss=False, stop_type='percent', stop_value=0.02)
    result_with_stop = analysis.simulate_trading(stop_loss=True, stop_type='percent', stop_value=0.02)
    result_bh = analysis.simulate_buy_and_hold()

    # Armazena todos os resultados
    results[stock] = {
        'no_stop_loss': result_no_stop,
        'with_stop_loss': result_with_stop,
        'buy_and_hold': result_bh
    }

# Display results
for stock, result in results.items():
    print(f"\nResults for {stock}:")
    print(f"No Stop Loss: {result['no_stop_loss']}")
    print(f"With Stop Loss: {result['with_stop_loss']}")
    print(f"Buy and Hold: {result['buy_and_hold']}")


Analyzing stock: ITUB4
Model loaded from ../../../saved_models/ITUB4_model_v1.1.pkl
Scaler loaded from ../../../saved_models/ITUB4_scaler_X_v1.1.pkl
Analyzing stock: VALE3
Model loaded from ../../../saved_models/VALE3_model_v1.1.pkl
Scaler loaded from ../../../saved_models/VALE3_scaler_X_v1.1.pkl

Results for ITUB4:
No Stop Loss: {'total_return': 0.004306758880615234, 'hit_rate': 0.5091693635382956, 'sharpe_ratio': 0.009284510031766181, 'max_drawdown': 0.011019912556144207, 'final_capital': 100430.67588806152, 'total_trades': 927, 'stop_triggered': 0}
With Stop Loss: {'total_return': 0.04480917984008789, 'hit_rate': 0.5091693635382956, 'sharpe_ratio': 0.11219380870429702, 'max_drawdown': 0.004647581876944222, 'final_capital': 104480.91798400879, 'total_trades': 927, 'stop_triggered': 133}
Buy and Hold: {'total_return': -0.004306758880615241, 'initial_price': 28.61874961853028, 'final_price': 24.31199073791504, 'final_capital': 99569.32411193848, 'shares_held': 100, 'days_held': 928}

R